# CATH — Protein Domain Structure Classification

**CATH** is a hierarchical classification of protein domain structures derived from the Protein Data Bank (PDB). Each domain is assigned to one of four levels:

| Level | Name | Description |
|---|---|---|
| **C** | Class | Secondary-structure composition (mainly-α, mainly-β, mixed α/β, few SS) |
| **A** | Architecture | Overall shape of the domain (e.g. barrel, sandwich, horseshoe) |
| **T** | Topology (fold) | Sequential connectivity of secondary structures |
| **H** | Homologous superfamily | Domains sharing a common ancestor inferred by sequence and structure |

Superfamily IDs take the form `C.A.T.H` (e.g. `1.10.510.10` — globins). CATH currently classifies **>500,000 domain structures** from **>150,000 PDB entries** into **~6,400 superfamilies**.

Key data types:
| Field | Description |
|---|---|
| `superfamily_id` | Dot-notation CATH ID (C.A.T.H) |
| `domain_count` | Number of classified domain structures |
| `pdb_count` | Number of PDB entries contributing domains |
| `domain_id` | Unique domain identifier (PDB + chain + segment, e.g. `1a3nA00`) |

**API base:** `https://www.cathdb.info/version/v4_3_0/api/rest/`

**Reference:** Sillitoe et al. (2021), *Nucleic Acids Research*, CATH v4.3

# TODO

* [x] **Ingest data**
    * [x] Connect to CATH REST API and confirm access (fetch superfamily `1.10.10.10`)
    * [x] Fetch all superfamilies (paginated, cache to `data/cath_superfamilies.json`)
    * [x] Parse into a Polars DataFrame: `superfamily_id`, class, architecture, topology, homologous superfamily, domain count, PDB count
    * [x] Fetch domains for a well-studied superfamily (`1.10.510.10` — globins) into a domains DataFrame
    * [x] Print shape, dtypes, and head for both DataFrames
* [ ] **Explore and clean**
    * [ ] Summarize superfamily size distributions (domain count, PDB count)
    * [ ] Examine class/architecture composition across the hierarchy
    * [ ] Handle any missing or malformed fields
* [ ] **Analysis**
    * [ ] Identify the largest and most diverse superfamilies
    * [ ] Analyse domain length distributions within a superfamily
    * [ ] Compare structural diversity across CATH classes
* [ ] **Visualization**
    * [ ] Sunburst / treemap of the CATH hierarchy (C → A → T → H)
    * [ ] Scatter plot of domain count vs PDB count per superfamily
    * [ ] Residue-range plot for domains within the globin superfamily
* [ ] **Statistical analysis**
    * [ ] Discuss the mathematical basis of structural comparison metrics (TM-score, RMSD)
    * [ ] Test whether domain counts are power-law distributed (a common observation in protein families)
    * [ ] Multiple hypothesis correction considerations for large-scale structural comparisons

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to CATH API and Confirm Access

In [ ]:
CATH_BASE = "https://www.cathdb.info/version/v4_3_0/api/rest"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def cath_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the CATH REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to CATH_BASE (e.g. "superfamily/1.10.10.10").
    params : dict, optional
        Query parameters (e.g. pagination start/stop).

    Returns
    -------
    dict
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{CATH_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=30)
    resp.raise_for_status()
    return resp.json()


# Connectivity check: fetch the well-studied winged-helix superfamily 1.10.10.10
sample = cath_get("superfamily/1.10.10.10")

# The CATH API wraps every response under a top-level key that mirrors the resource
data = sample.get("data", {})
print(f"Superfamily ID  : {data.get('superfamily_id', data.get('cathCode', 'n/a'))}")
print(f"Name            : {data.get('name', 'n/a')}")
print(f"Domain count    : {data.get('numDomains', data.get('domain_count', 'n/a'))}")
print(f"Classification  : Class {data.get('classNumber', '?')} | "
      f"Arch {data.get('archNumber', '?')} | "
      f"Topo {data.get('topoNumber', '?')} | "
      f"Homsuper {data.get('homologousSuperfamilyNumber', '?')}")
print("\nFull response keys:", list(data.keys()))

### 1.2 Fetch All Superfamilies (Paginated)

In [ ]:
SUPERFAMILIES_CACHE = DATA_DIR / "cath_superfamilies.json"
PAGE_SIZE = 100   # records per page; CATH accepts start/stop parameters


def fetch_all_superfamilies(cache_path: Path = SUPERFAMILIES_CACHE) -> list[dict]:
    """
    Fetch the complete list of CATH superfamilies via the paginated
    ``/superfamily`` endpoint, caching results to disk.

    The endpoint accepts ``start`` (0-indexed offset) and ``stop``
    (exclusive upper bound) query parameters. We page through in steps
    of PAGE_SIZE until an empty batch is returned.

    Parameters
    ----------
    cache_path : Path
        File path for the JSON cache. Skips download if the file exists.

    Returns
    -------
    list[dict]
        One dict per superfamily with all fields returned by the API.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_sfams = []
    start = 0

    while True:
        resp = cath_get("superfamily", {"start": start, "stop": start + PAGE_SIZE})

        # The API returns results under 'data'; fall back to a list at root
        batch = resp.get("data", resp) if isinstance(resp, dict) else resp
        if isinstance(batch, dict):
            # Some CATH versions wrap in {"superfamilies": [...]}
            batch = batch.get("superfamilies", batch.get("data", []))

        if not batch:
            break   # empty page → we have everything

        all_sfams.extend(batch)
        print(f"  Fetched {len(all_sfams)} superfamilies so far ...", end="\r")

        # If the batch is smaller than PAGE_SIZE, this was the last page
        if len(batch) < PAGE_SIZE:
            break

        start += PAGE_SIZE
        time.sleep(0.3)   # polite delay between paginated requests

    print(f"\nTotal superfamilies fetched: {len(all_sfams)}")
    cache_path.write_text(json.dumps(all_sfams))
    print(f"Cached to: {cache_path}")
    return all_sfams


superfamilies_raw = fetch_all_superfamilies()
print(f"\nSample record keys: {list(superfamilies_raw[0].keys()) if superfamilies_raw else 'n/a'}")
print(f"First record: {superfamilies_raw[0] if superfamilies_raw else 'n/a'}")

### 1.3 Parse into a Polars DataFrame

In [ ]:
def parse_superfamily(record: dict) -> dict:
    """
    Flatten a single raw superfamily record into a row-friendly dict.

    The CATH API may return the superfamily ID either as a dot-notation
    string (``cathCode`` / ``superfamily_id``) or as four separate integer
    fields.  We normalise both layouts here.

    Parameters
    ----------
    record : dict
        Raw superfamily record from the CATH ``/superfamily`` endpoint.

    Returns
    -------
    dict
        Flat dict with typed scalar values suitable for a Polars DataFrame row.
    """
    # Superfamily ID can arrive as "cathCode", "superfamilyId", or "id"
    sfam_id = (
        record.get("cathCode")
        or record.get("superfamilyId")
        or record.get("superfamily_id")
        or record.get("id")
        or ""
    )

    # Split the dot-notation ID into its four integer levels
    parts = sfam_id.split(".") if sfam_id else ["0", "0", "0", "0"]
    while len(parts) < 4:
        parts.append("0")

    return {
        "superfamily_id":        sfam_id,
        "class":                 int(parts[0]) if parts[0].isdigit() else None,
        "architecture":          int(parts[1]) if parts[1].isdigit() else None,
        "topology":              int(parts[2]) if parts[2].isdigit() else None,
        "homologous_superfamily": int(parts[3]) if parts[3].isdigit() else None,
        # Domain and PDB counts — field names vary across API versions
        "domain_count": (
            record.get("numDomains")
            or record.get("domain_count")
            or record.get("num_domains")
        ),
        "pdb_count": (
            record.get("numPdbs")
            or record.get("pdb_count")
            or record.get("num_pdbs")
        ),
    }


rows = [parse_superfamily(r) for r in superfamilies_raw]

superfamilies = pl.DataFrame(rows).with_columns([
    pl.col("class").cast(pl.Int32),
    pl.col("architecture").cast(pl.Int32),
    pl.col("topology").cast(pl.Int32),
    pl.col("homologous_superfamily").cast(pl.Int32),
    pl.col("domain_count").cast(pl.Int32, strict=False),   # may be null
    pl.col("pdb_count").cast(pl.Int32, strict=False),
])

print(f"Shape  : {superfamilies.shape}")
print(f"Memory : {superfamilies.estimated_size('kb'):.1f} KB")
print("\nDtypes:")
print(superfamilies.schema)
print()
superfamilies.head(10)

### 1.4 Fetch Domains for a Well-Studied Superfamily (Globins — 1.10.510.10)

In [ ]:
# Globins (1.10.510.10) are the canonical example of the globin fold — an
# all-α bundle that binds haem and transports oxygen.  With hundreds of
# classified structures this superfamily is ideal for a domain-level overview.
GLOBIN_SFAM = "1.10.510.10"
DOMAINS_CACHE = DATA_DIR / f"cath_domains_{GLOBIN_SFAM.replace('.', '_')}.json"


def fetch_superfamily_domains(sfam_id: str, cache_path: Path) -> list[dict]:
    """
    Fetch all domain records for a CATH superfamily from the
    ``/superfamily/{sfam_id}/domain`` endpoint, with disk caching.

    Parameters
    ----------
    sfam_id : str
        Dot-notation CATH superfamily ID (e.g. ``"1.10.510.10"``).
    cache_path : Path
        File path for the JSON cache.

    Returns
    -------
    list[dict]
        One dict per domain; fields include domain_id, pdb_id, chain_id,
        and segment start/stop residues.
    """
    if cache_path.exists():
        print(f"Loading domains from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    resp = cath_get(f"superfamily/{sfam_id}/domain")
    # Domains are returned under "data" (list) or directly as a list
    domains = resp.get("data", resp) if isinstance(resp, dict) else resp
    if isinstance(domains, dict):
        domains = domains.get("domains", [])

    print(f"Fetched {len(domains)} domains for superfamily {sfam_id}")
    cache_path.write_text(json.dumps(domains))
    return domains


domains_raw = fetch_superfamily_domains(GLOBIN_SFAM, DOMAINS_CACHE)
print(f"Sample domain record: {domains_raw[0] if domains_raw else 'n/a'}")

In [ ]:
def parse_domain(record: dict) -> dict:
    """
    Flatten a single raw domain record into a row-friendly dict.

    CATH domain IDs follow the convention ``{pdb}{chain}{segment}``,
    e.g. ``1a3nA00`` (PDB 1a3n, chain A, segment 00).  Segment boundaries
    (start_res, stop_res) are provided either at the top level or nested
    under a ``segments`` list.

    Parameters
    ----------
    record : dict
        Raw domain record from ``/superfamily/{sfam_id}/domain``.

    Returns
    -------
    dict
        Flat dict with domain_id, pdb_id, chain_id, start_res, stop_res.
    """
    domain_id = (
        record.get("domain_id")
        or record.get("domainId")
        or record.get("id")
        or ""
    )

    # PDB ID and chain are the first 4 and 5th characters of the domain ID
    pdb_id  = domain_id[:4].lower() if len(domain_id) >= 4 else record.get("pdbId", "")
    chain_id = domain_id[4].upper() if len(domain_id) >= 5 else record.get("chainId", "")

    # Residue ranges may be at the top level or inside a segments list
    segments = record.get("segments") or []
    if segments:
        # Take the first segment's boundaries (domains may span multiple segments)
        start_res = segments[0].get("start", segments[0].get("startRes"))
        stop_res  = segments[0].get("stop",  segments[0].get("stopRes"))
    else:
        start_res = record.get("startRes") or record.get("start_res")
        stop_res  = record.get("stopRes")  or record.get("stop_res")

    return {
        "domain_id": domain_id,
        "pdb_id":    pdb_id,
        "chain_id":  chain_id,
        "start_res": start_res,
        "stop_res":  stop_res,
    }


domain_rows = [parse_domain(r) for r in domains_raw]

domains = pl.DataFrame(domain_rows).with_columns([
    pl.col("start_res").cast(pl.Int32, strict=False),
    pl.col("stop_res").cast(pl.Int32, strict=False),
])

# Derive domain length where both boundaries are present
domains = domains.with_columns(
    (pl.col("stop_res") - pl.col("start_res") + 1).alias("domain_length")
)

print(f"Shape  : {domains.shape}")
print(f"\nDtypes:")
print(domains.schema)
print()
domains.head(10)

### 1.5 Summary

In [ ]:
print("=" * 55)
print("superfamilies DataFrame")
print("=" * 55)
print(f"  Shape  : {superfamilies.shape[0]:,} rows × {superfamilies.shape[1]} columns")
print(f"  Dtypes : {dict(superfamilies.schema)}")
print()
print(superfamilies.head(5))

print()
print("=" * 55)
print(f"domains DataFrame  (superfamily {GLOBIN_SFAM} — globins)")
print("=" * 55)
print(f"  Shape  : {domains.shape[0]:,} rows × {domains.shape[1]} columns")
print(f"  Dtypes : {dict(domains.schema)}")
print()
print(domains.head(5))